<a href="https://colab.research.google.com/github/RachmadAprisandhy/PembelajaranMesin/blob/main/JS09/JS09_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

install library

In [1]:
!pip install -q annoy faiss-cpu hnswlib scikit-learn pandas numpy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


upload data dari lokal llau baca dataset

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
dataset_path = '/content/drive/MyDrive/Pembelajaran mesin/songs_with_attributes_and_lyrics.csv'

Load dataset

In [17]:
import pandas as pd
import numpy as np

features = [
    'danceability',
    'energy',
    'loudness',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo'
]

# Periksa nama kolom
header = pd.read_csv(dataset_path, nrows=0)

print("Kolom dataset:")
print(header.columns.tolist())

# Ambil metadata lagu jika tersedia
metadata = [
    col for col in ['track_name', 'track_artist']
    if col in header.columns
]

# Load hanya fitur yang diperlukan
df = pd.read_csv(
    dataset_path,
    usecols=features + metadata
)

print("Ukuran dataset:", df.shape)
display(df.head())

Kolom dataset:
['id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics']
Ukuran dataset: (955320, 9)


,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0.415,0.6050,-11.157,0.0575,0.00116,0.838000,0.4710,0.193,100.059
1,0.788,0.6480,-9.135,0.3150,0.90000,0.000000,0.1760,0.287,79.998
2,0.000,0.0354,-20.151,0.0000,0.90800,0.000000,0.4790,0.000,0.000
3,0.657,0.8820,-6.340,0.0385,0.00740,0.000013,0.0474,0.939,123.588
4,0.659,0.8930,-8.531,0.0411,0.09220,0.000019,0.0534,0.951,123.600


standard scaler

In [18]:
from sklearn.preprocessing import StandardScaler

df[features] = df[features].apply(
    pd.to_numeric,
    errors='coerce'
)

df = df.dropna(subset=features).reset_index(drop=True)

scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    df[features]
).astype(np.float32)

print("Jumlah data:", len(X_scaled))
print("Dimensi:", X_scaled.shape)
print("StandardScaler berhasil!")

Jumlah data: 955320
Dimensi: (955320, 9)
StandardScaler berhasil!


kode uji

In [7]:
# ==========================================
# EKSPERIMEN LAB 2 - SPOTIFY DATASET
# ==========================================

import time
import numpy as np
import pandas as pd

from sklearn.neighbors import NearestNeighbors
from annoy import AnnoyIndex
import faiss
import hnswlib

# Parameter
k = 10
n_queries = min(100, len(X_scaled))

# Query yang sama untuk semua algoritma
rng = np.random.default_rng(42)

query_indices = rng.choice(
    len(X_scaled),
    size=n_queries,
    replace=False
)

Q = X_scaled[query_indices]
dim = X_scaled.shape[1]
n_data = len(X_scaled)

results = []
neighbors_results = {}

print("Jumlah data:", n_data)
print("Dimensi:", dim)
print("Jumlah query:", n_queries)


# ==========================================
# FUNGSI RECALL
# ==========================================

def calculate_recall(approx_results, exact_results):
    recalls = []

    for approx, exact in zip(approx_results, exact_results):
        score = len(set(approx) & set(exact)) / k
        recalls.append(score)

    return np.mean(recalls) * 100


def remove_self(ids, query_ids):
    return [
        [int(x) for x in row if int(x) != int(q)][:k]
        for row, q in zip(ids, query_ids)
    ]


# ==========================================
# 1. EXACT NEAREST NEIGHBORS
# ==========================================

print("\nRunning Exact NN...")

exact_index = NearestNeighbors(
    n_neighbors=k + 1,
    algorithm='brute',
    metric='euclidean',
    n_jobs=-1
)

start = time.perf_counter()
exact_index.fit(X_scaled)
exact_build = time.perf_counter() - start

start = time.perf_counter()
exact_distances, exact_ids = exact_index.kneighbors(Q)
exact_query = time.perf_counter() - start

exact_neighbors = remove_self(
    exact_ids, query_indices
)

neighbors_results['Exact NN'] = exact_neighbors

results.append({
    'Algorithm': 'Exact NN',
    'Build (s)': exact_build,
    'Query (ms)': exact_query / n_queries * 1000,
    'Recall@10 (%)': 100.0
})


# ==========================================
# 2. ANNOY
# ==========================================

print("Running Annoy...")

annoy_index = AnnoyIndex(dim, 'euclidean')

start = time.perf_counter()

for i in range(n_data):
    annoy_index.add_item(i, X_scaled[i])

annoy_index.build(10)

annoy_build = time.perf_counter() - start

start = time.perf_counter()

annoy_ids = [
    annoy_index.get_nns_by_vector(
        q.tolist(), k + 1
    )
    for q in Q
]

annoy_query = time.perf_counter() - start

annoy_neighbors = remove_self(
    annoy_ids, query_indices
)

neighbors_results['ANNOY'] = annoy_neighbors

results.append({
    'Algorithm': 'ANNOY',
    'Build (s)': annoy_build,
    'Query (ms)': annoy_query / n_queries * 1000,
    'Recall@10 (%)': calculate_recall(
        annoy_neighbors, exact_neighbors
    )
})


# ==========================================
# 3. FAISS - IVF FLAT
# ==========================================

print("Running FAISS...")

nlist = min(1024, max(1, int(np.sqrt(n_data))))

quantizer = faiss.IndexFlatL2(dim)

faiss_index = faiss.IndexIVFFlat(
    quantizer,
    dim,
    nlist,
    faiss.METRIC_L2
)

start = time.perf_counter()

train_size = min(n_data, max(10000, nlist * 40))

train_rng = np.random.default_rng(42)

train_ids = train_rng.choice(
    n_data,
    size=train_size,
    replace=False
)

faiss_index.train(X_scaled[train_ids])
faiss_index.add(X_scaled)

faiss_build = time.perf_counter() - start

faiss_index.nprobe = min(10, nlist)

start = time.perf_counter()

faiss_distances, faiss_ids = faiss_index.search(
    Q, k + 1
)

faiss_query = time.perf_counter() - start

faiss_neighbors = remove_self(
    faiss_ids, query_indices
)

neighbors_results['FAISS'] = faiss_neighbors

results.append({
    'Algorithm': 'FAISS',
    'Build (s)': faiss_build,
    'Query (ms)': faiss_query / n_queries * 1000,
    'Recall@10 (%)': calculate_recall(
        faiss_neighbors, exact_neighbors
    )
})


# ==========================================
# 4. HNSW
# ==========================================

print("Running HNSW...")

hnsw_index = hnswlib.Index(
    space='l2',
    dim=dim
)

start = time.perf_counter()

hnsw_index.init_index(
    max_elements=n_data,
    ef_construction=200,
    M=16
)

hnsw_index.add_items(
    X_scaled,
    num_threads=-1
)

hnsw_build = time.perf_counter() - start

hnsw_index.set_ef(50)

start = time.perf_counter()

hnsw_ids, hnsw_distances = hnsw_index.knn_query(
    Q, k=k + 1
)

hnsw_query = time.perf_counter() - start

hnsw_neighbors = remove_self(
    hnsw_ids, query_indices
)

neighbors_results['HNSW'] = hnsw_neighbors

results.append({
    'Algorithm': 'HNSW',
    'Build (s)': hnsw_build,
    'Query (ms)': hnsw_query / n_queries * 1000,
    'Recall@10 (%)': calculate_recall(
        hnsw_neighbors, exact_neighbors
    )
})

print("\nSEMUA EKSPERIMEN SELESAI!")

Jumlah data: 955320
Dimensi: 9
Jumlah query: 100

Running Exact NN...
Running Annoy...
Running FAISS...
Running HNSW...

SEMUA EKSPERIMEN SELESAI!


table perbandingan

In [8]:
df_results = pd.DataFrame(results)

display(
    df_results.style.format({
        'Build (s)': '{:.4f}',
        'Query (ms)': '{:.4f}',
        'Recall@10 (%)': '{:.2f}%'
    })
)

,Algorithm,Build (s),Query (ms),Recall@10 (%)
0,Exact NN,0.1792,19.5442,100.00%
1,ANNOY,25.5828,0.0583,84.90%
2,FAISS,0.6973,0.1508,99.10%
3,HNSW,208.9438,0.0554,99.50%


10 lagu terdekat

In [9]:
selected_track = int(query_indices[0])

metadata_cols = [
    col for col in [
        'track_name',
        'track_artist'
    ]
    if col in df.columns
]

print("LAGU YANG DIPILIH:")

display(df.iloc[[selected_track]][metadata_cols])

for algorithm, neighbor_ids in neighbors_results.items():

    print(f"\n=== HASIL {algorithm} ===")

    nearest_tracks = df.iloc[
        neighbor_ids[0]
    ][metadata_cols].copy()

    nearest_tracks.insert(
        0, 'Rank', range(1, len(nearest_tracks) + 1)
    )

    display(nearest_tracks)

LAGU YANG DIPILIH:


""
417612



=== HASIL Exact NN ===


,Rank
417613,1
345421,2
345420,3
55790,4
847343,5
228789,6
628101,7
2642,8
199711,9
231313,10



=== HASIL ANNOY ===


,Rank
417613,1
345420,2
345421,3
55790,4
628101,5
231313,6
101638,7
326195,8
952165,9
720147,10



=== HASIL FAISS ===


,Rank
417613,1
345420,2
345421,3
55790,4
847343,5
228789,6
628101,7
2642,8
199711,9
231313,10



=== HASIL HNSW ===


,Rank
417613,1
345420,2
345421,3
55790,4
847343,5
228789,6
628101,7
2642,8
199711,9
231313,10


In [10]:
print(df.columns.tolist())

['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']


In [11]:
display(
    df.iloc[[selected_track]]
)

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
417612,0.618,0.925,-6.209,0.102,0.163,0.0,0.268,0.739,175.117


In [12]:
header = pd.read_csv(dataset_path, nrows=0)

print(header.columns.tolist())

['id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics']
